# 📐 EFP Term Structure — XAU · XAG · XPT · XPD
**Bid/Ask implied EFP across C1–C4 | Annualised to First Notice Date + 1**

---
| Field | Definition |
|---|---|
| **EFP Bid** | Futures Bid − Spot Ask  *(sell EFP: sell spot at ask, buy futures at bid)* |
| **EFP Ask** | Futures Ask − Spot Bid  *(buy EFP: buy spot at bid, sell futures at ask)* |
| **EFP Mid** | (EFP Bid + EFP Ask) / 2 |
| **Ann. EFP** | EFP / Spot Mid × 360 / Days × 100  *(days: spot T+2 → FND + 1 biz day)* |


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELL 1 — Imports, BQL Service, Styling
# ─────────────────────────────────────────────────────────────
import bql
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display, HTML
from datetime import datetime, date, timedelta
import warnings
warnings.filterwarnings('ignore')

# ── Colour palette ────────────────────────────────────────────
BG_DARK  = '#0d1117'
BG_PANEL = '#161b22'
BG_CARD  = '#1c2128'
C_GOLD   = '#FFD700'
C_SILVER = '#C0C0C0'
C_PLAT   = '#58a6ff'
C_PALL   = '#3fb950'
C_RED    = '#f85149'
C_GREEN  = '#3fb950'
C_AMBER  = '#d29922'
C_MUTED  = '#8b949e'
C_TEXT   = '#e6edf3'
C_GRID   = '#30363d'

METAL_COLORS = {
    'XAU': C_GOLD,
    'XAG': C_SILVER,
    'XPT': C_PLAT,
    'XPD': C_PALL,
}
METAL_NAMES = {
    'XAU': 'Gold',
    'XAG': 'Silver',
    'XPT': 'Platinum',
    'XPD': 'Palladium',
}

plt.rcParams.update({
    'figure.facecolor': BG_DARK,  'axes.facecolor':  BG_PANEL,
    'axes.edgecolor':   C_GRID,   'axes.labelcolor': C_MUTED,
    'xtick.color':      C_MUTED,  'ytick.color':     C_MUTED,
    'text.color':       C_TEXT,   'grid.color':      C_GRID,
    'grid.linestyle':   '--',     'grid.alpha':      0.5,
    'lines.linewidth':  1.8,      'font.family':     'monospace',
    'font.size':        9,        'axes.titlesize':  10,
    'axes.titlecolor':  C_TEXT,   'axes.titleweight':'bold',
    'figure.dpi':       120,      'legend.framealpha':0.3,
    'legend.facecolor': BG_CARD,  'legend.edgecolor': C_GRID,
})

bq = bql.Service()
TODAY = date.today()
T_STR = TODAY.strftime('%Y-%m-%d')
print(f"✅  BQL Service ready | {datetime.now().strftime('%Y-%m-%d  %H:%M:%S')}")


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELL 2 — Ticker Config
# ─────────────────────────────────────────────────────────────

# Spot tickers (BGN composite — bid/ask available)
SPOT_TICKERS = {
    'XAU': 'XAUUSD Curncy',
    'XAG': 'XAGUSD Curncy',
    'XPT': 'XPTUSD Curncy',
    'XPD': 'XPDUSD Curncy',
}

# Futures continuous contracts C1–C4 (BQuant roll codes)
FUTURES_CHAIN = {
    'XAU': {'C1': 'GCA Comdty', 'C2': 'GCB Comdty', 'C3': 'GCC Comdty', 'C4': 'GCD Comdty'},
    'XAG': {'C1': 'SIA Comdty', 'C2': 'SIB Comdty', 'C3': 'SIC Comdty', 'C4': 'SID Comdty'},
    'XPT': {'C1': 'PLA Comdty', 'C2': 'PLB Comdty', 'C3': 'PLC Comdty', 'C4': 'PLD Comdty'},
    'XPD': {'C1': 'PAA Comdty', 'C2': 'PAB Comdty', 'C3': 'PAC Comdty', 'C4': 'PAD Comdty'},
}

CONTRACTS = ['C1', 'C2', 'C3', 'C4']
METALS    = ['XAU', 'XAG', 'XPT', 'XPD']

print("✅  Config loaded")
for m in METALS:
    tkrs = ' | '.join(f"{k}: {v}" for k, v in FUTURES_CHAIN[m].items())
    print(f"  {m}  spot: {SPOT_TICKERS[m]}   futures: {tkrs}")


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELL 3 — Fetch Bid / Ask
# ─────────────────────────────────────────────────────────────

def fetch_bid_ask(tickers):
    """Fetch PX_BID and PX_ASK for a list of tickers.
    Returns dict: {ticker: {'bid': float, 'ask': float, 'mid': float}}"""
    if isinstance(tickers, str):
        tickers = [tickers]
    tkr_str = ', '.join(f"'{t}'" for t in tickers)
    result = {t: {'bid': np.nan, 'ask': np.nan, 'mid': np.nan} for t in tickers}
    try:
        resp_bid = bq.execute(f"get('PX_BID') for([{tkr_str}])")
        resp_ask = bq.execute(f"get('PX_ASK') for([{tkr_str}])")
        df_bid = resp_bid[0].df()
        df_ask = resp_ask[0].df()
        for t in tickers:
            try:
                bid = float(df_bid.loc[t, 'PX_BID'])
                ask = float(df_ask.loc[t, 'PX_ASK'])
                result[t] = {'bid': bid, 'ask': ask, 'mid': (bid + ask) / 2}
            except Exception:
                pass
    except Exception as e:
        print(f"  ⚠ fetch_bid_ask error: {e}")
    return result


def fetch_fnd(tickers):
    """Fetch FUT_NOTICE_FIRST. Returns dict {ticker: date}."""
    tkr_str = ', '.join(f"'{t}'" for t in tickers)
    result = {}
    try:
        resp = bq.execute(f"get('FUT_NOTICE_FIRST') for([{tkr_str}])")
        df = resp[0].df()
        for t in tickers:
            try:
                result[t] = pd.Timestamp(df.loc[t, 'FUT_NOTICE_FIRST']).date()
            except Exception:
                result[t] = None
    except Exception as e:
        print(f"  ⚠ fetch_fnd error: {e}")
    return result


# ── Collect all tickers ───────────────────────────────────────
all_spot_tickers = list(SPOT_TICKERS.values())
all_fut_tickers  = [tkr for m in METALS for tkr in FUTURES_CHAIN[m].values()]

print("Fetching spot bid/ask...")
spot_ba = fetch_bid_ask(all_spot_tickers)

print("Fetching futures bid/ask...")
fut_ba = fetch_bid_ask(all_fut_tickers)

print("Fetching first notice dates...")
fnd_map = fetch_fnd(all_fut_tickers)

# ── Spot date T+2 ─────────────────────────────────────────────
spot_date = pd.bdate_range(start=TODAY, periods=3)[-1].date()
print(f"\nSpot date (T+2 biz): {spot_date}")

# ── Summary ───────────────────────────────────────────────────
print(f"\n{'Metal':<6}  {'Spot Bid':>10}  {'Spot Ask':>10}  {'Spot Mid':>10}")
print(f"{'─'*6}  {'─'*10}  {'─'*10}  {'─'*10}")
for metal, tkr in SPOT_TICKERS.items():
    ba = spot_ba[tkr]
    print(f"{metal:<6}  {ba['bid']:>10.4f}  {ba['ask']:>10.4f}  {ba['mid']:>10.4f}")


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELL 4 — EFP Calculation & Tables
#
#  EFP Bid = Futures Bid − Spot Ask
#  EFP Ask = Futures Ask − Spot Bid
#  EFP Mid = (EFP Bid + EFP Ask) / 2
#  Ann.    = EFP / Spot Mid × 360 / Days × 100
#  Days    = spot_date → FND + 1 biz day
# ─────────────────────────────────────────────────────────────

def days_to_value(fnd_date, spot_dt):
    """Days from spot_dt to FND + 1 business day (CME delivery value date)."""
    if fnd_date is None:
        return None
    fnd_value = pd.bdate_range(start=fnd_date, periods=2)[-1].date()
    return max((fnd_value - spot_dt).days, 1)


efp_data = {}   # {metal: DataFrame}

for metal in METALS:
    spot_tkr = SPOT_TICKERS[metal]
    sba = spot_ba[spot_tkr]
    spot_bid = sba['bid']
    spot_ask = sba['ask']
    spot_mid = sba['mid']

    rows = []
    for contract, fut_tkr in FUTURES_CHAIN[metal].items():
        fba  = fut_ba[fut_tkr]
        fnd  = fnd_map.get(fut_tkr)
        days = days_to_value(fnd, spot_date)

        fut_bid = fba['bid']
        fut_ask = fba['ask']
        fut_mid = fba['mid']

        efp_bid = fut_bid - spot_ask if not (np.isnan(fut_bid) or np.isnan(spot_ask)) else np.nan
        efp_ask = fut_ask - spot_bid if not (np.isnan(fut_ask) or np.isnan(spot_bid)) else np.nan
        efp_mid = (efp_bid + efp_ask) / 2 if not (np.isnan(efp_bid) or np.isnan(efp_ask)) else np.nan
        spread  = efp_ask - efp_bid  if not (np.isnan(efp_ask) or np.isnan(efp_bid)) else np.nan

        def ann(efp_val):
            if days and not (np.isnan(efp_val) or np.isnan(spot_mid)) and spot_mid != 0:
                return efp_val / spot_mid * (360 / days) * 100
            return np.nan

        rows.append({
            'Contract'     : contract,
            'Fut Tkr'      : fut_tkr,
            'FND'          : str(fnd) if fnd else '—',
            'Days'         : days or '—',
            'Fut Bid'      : fut_bid,
            'Fut Ask'      : fut_ask,
            'Spot Bid'     : spot_bid,
            'Spot Ask'     : spot_ask,
            'EFP Bid'      : efp_bid,
            'EFP Ask'      : efp_ask,
            'EFP Mid'      : efp_mid,
            'EFP Spread'   : spread,
            'Ann Bid (%)'  : ann(efp_bid),
            'Ann Ask (%)'  : ann(efp_ask),
            'Ann Mid (%)'  : ann(efp_mid),
        })

    efp_data[metal] = pd.DataFrame(rows).set_index('Contract')

# ── Display tables ────────────────────────────────────────────
BAR = '═' * 70

for metal in METALS:
    df = efp_data[metal]
    color = METAL_COLORS[metal]
    name  = METAL_NAMES[metal]
    spot_mid = spot_ba[SPOT_TICKERS[metal]]['mid']

    print(f"\n{BAR}")
    print(f"  {name} ({metal})   Spot Mid: ${spot_mid:,.4f}")
    print(BAR)

    display_cols = ['Days', 'Fut Bid', 'Fut Ask',
                    'EFP Bid', 'EFP Ask', 'EFP Mid', 'EFP Spread',
                    'Ann Bid (%)', 'Ann Ask (%)', 'Ann Mid (%)']

    def _color_efp(v):
        if not isinstance(v, float) or np.isnan(v):
            return ''
        return f'color:{C_GREEN}' if v > 0 else f'color:{C_RED}'

    styler = (df[display_cols].style
        .format({
            'Fut Bid'    : '{:,.4f}',
            'Fut Ask'    : '{:,.4f}',
            'EFP Bid'    : '{:+.4f}',
            'EFP Ask'    : '{:+.4f}',
            'EFP Mid'    : '{:+.4f}',
            'EFP Spread' : '{:.4f}',
            'Ann Bid (%)': '{:+.4f}%',
            'Ann Ask (%)': '{:+.4f}%',
            'Ann Mid (%)': '{:+.4f}%',
        }, na_rep='—')
        .applymap(_color_efp, subset=['EFP Bid','EFP Ask','EFP Mid',
                                       'Ann Bid (%)','Ann Ask (%)','Ann Mid (%)'])
        .set_caption(f'{name} EFP Term Structure  |  EFP Bid = Fut Bid − Spot Ask  |  EFP Ask = Fut Ask − Spot Bid')
        .set_table_styles([
            {'selector': 'caption', 'props': [('color', C_TEXT), ('font-weight', 'bold'), ('font-size', '11px')]},
            {'selector': 'th', 'props': [('background-color', BG_CARD), ('color', C_TEXT)]},
        ])
    )
    display(styler)


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELL 5 — Switch Levels
#
#  Switch N→N+1 = EFP(Cn+1) − EFP(Cn)
#
#  Bid/Ask crossed:
#    Switch Bid  = EFP Bid(Cn+1) − EFP Ask(Cn)
#                  (income received when rolling a short from Cn to Cn+1)
#    Switch Ask  = EFP Ask(Cn+1) − EFP Bid(Cn)
#                  (cost paid when rolling a long from Cn to Cn+1)
#    Switch Mid  = (Switch Bid + Switch Ask) / 2
#
#  Annualised over the INCREMENTAL period:
#    Days = Days(Cn+1 value) − Days(Cn value)
#    Ann  = Switch Mid / Spot Mid × 360 / Days × 100
# ─────────────────────────────────────────────────────────────

switch_data = {}   # {metal: DataFrame}

SWITCH_PAIRS = [('C1','C2'), ('C2','C3'), ('C3','C4')]

BAR = '═' * 70

for metal in METALS:
    spot_mid = spot_ba[SPOT_TICKERS[metal]]['mid']
    df       = efp_data[metal]

    rows = []
    for cn, cn1 in SWITCH_PAIRS:
        r_near = df.loc[cn]
        r_far  = df.loc[cn1]

        sw_bid = r_far['EFP Bid'] - r_near['EFP Ask']
        sw_ask = r_far['EFP Ask'] - r_near['EFP Bid']
        sw_mid = (sw_bid + sw_ask) / 2 if not (np.isnan(sw_bid) or np.isnan(sw_ask)) else np.nan
        sw_spd = sw_ask - sw_bid        if not (np.isnan(sw_ask) or np.isnan(sw_bid)) else np.nan

        # Incremental days between the two value dates
        days_near = r_near['Days'] if isinstance(r_near['Days'], int) else None
        days_far  = r_far['Days']  if isinstance(r_far['Days'],  int) else None
        inc_days  = (days_far - days_near) if (days_near and days_far) else None

        def ann_sw(val):
            if inc_days and inc_days > 0 and not np.isnan(val) and spot_mid and spot_mid != 0:
                return val / spot_mid * (360 / inc_days) * 100
            return np.nan

        rows.append({
            'Switch'         : f'{cn}→{cn1}',
            'Near FND'       : r_near['FND'],
            'Far FND'        : r_far['FND'],
            'Inc. Days'      : inc_days or '—',
            'Switch Bid'     : sw_bid,
            'Switch Ask'     : sw_ask,
            'Switch Mid'     : sw_mid,
            'Switch Spread'  : sw_spd,
            'Ann Bid (%)'    : ann_sw(sw_bid),
            'Ann Ask (%)'    : ann_sw(sw_ask),
            'Ann Mid (%)'    : ann_sw(sw_mid),
        })

    switch_data[metal] = pd.DataFrame(rows).set_index('Switch')

for metal in METALS:
    df    = switch_data[metal]
    color = METAL_COLORS[metal]
    name  = METAL_NAMES[metal]
    spot_mid = spot_ba[SPOT_TICKERS[metal]]['mid']

    print(f"\n{BAR}")
    print(f"  {name} ({metal})  Switch Levels   Spot Mid: ${spot_mid:,.4f}")
    print(BAR)

    display_cols = ['Inc. Days',
                    'Switch Bid', 'Switch Ask', 'Switch Mid', 'Switch Spread',
                    'Ann Bid (%)', 'Ann Ask (%)', 'Ann Mid (%)']

    def _col(v):
        if not isinstance(v, float) or np.isnan(v):
            return ''
        return f'color:{C_GREEN}' if v > 0 else f'color:{C_RED}'

    styler = (df[display_cols].style
        .format({
            'Switch Bid'   : '{:+.4f}',
            'Switch Ask'   : '{:+.4f}',
            'Switch Mid'   : '{:+.4f}',
            'Switch Spread': '{:.4f}',
            'Ann Bid (%)'  : '{:+.4f}%',
            'Ann Ask (%)'  : '{:+.4f}%',
            'Ann Mid (%)'  : '{:+.4f}%',
        }, na_rep='—')
        .applymap(_col, subset=['Switch Bid','Switch Ask','Switch Mid',
                                 'Ann Bid (%)','Ann Ask (%)','Ann Mid (%)'])
        .set_caption(
            f'{name} Switch Levels  |  '
            f'Bid = EFP Bid(far) − EFP Ask(near)  |  '
            f'Ann over incremental period between delivery dates')
        .set_table_styles([
            {'selector': 'caption', 'props': [('color', C_TEXT), ('font-weight','bold'), ('font-size','11px')]},
            {'selector': 'th',      'props': [('background-color', BG_CARD), ('color', C_TEXT)]},
        ])
    )
    display(styler)


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELL 6 — Switch Level Charts
#  Left:  absolute switch mid ($/oz) per metal
#  Right: annualised switch mid (%) per metal
# ─────────────────────────────────────────────────────────────

switch_labels = [f'{a}→{b}' for a, b in SWITCH_PAIRS]
x = np.arange(len(SWITCH_PAIRS))
w = 0.18
offsets = np.linspace(-(len(METALS)-1)*w/2, (len(METALS)-1)*w/2, len(METALS))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5), facecolor=BG_DARK)
fig.suptitle('EFP Switch Levels — Bid/Ask Mid', color=C_TEXT, fontsize=12, fontweight='bold')

for i, metal in enumerate(METALS):
    df     = switch_data[metal]
    color  = METAL_COLORS[metal]
    name   = METAL_NAMES[metal]

    mids     = df['Switch Mid'].tolist()
    ann_bids = df['Ann Bid (%)'].tolist()
    ann_asks = df['Ann Ask (%)'].tolist()
    ann_mids = df['Ann Mid (%)'].tolist()

    # ── Absolute switch mid ───────────────────────────────────
    ax1.bar(x + offsets[i], mids, width=w, color=color, alpha=0.8, label=name)

    # ── Annualised: band + mid line ───────────────────────────
    ax2.fill_between(x + offsets[i], ann_bids, ann_asks,
                     color=color, alpha=0.15)
    ax2.plot(x + offsets[i], ann_mids, color=color,
             marker='o', ms=5, lw=0, label=name)

# ── Formatting ────────────────────────────────────────────────
ax1.set_xticks(x); ax1.set_xticklabels(switch_labels)
ax1.axhline(0, color=C_MUTED, lw=0.8, ls=':')
ax1.set_title('Switch Mid ($/oz)', color=C_TEXT)
ax1.set_ylabel('$/oz', color=C_MUTED)
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3, axis='y')

ax2.set_xticks(x); ax2.set_xticklabels(switch_labels)
ax2.axhline(0, color=C_MUTED, lw=0.8, ls=':')
ax2.set_title('Ann. Switch Mid (%)  with Bid/Ask band', color=C_TEXT)
ax2.set_ylabel('Ann. (%)', color=C_MUTED)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:+.3f}%'))
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELL 5 — EFP Term Structure Charts
#  2×2 grid — one panel per metal
#  Bid/Ask band as filled area, Mid as line
# ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(16, 10), facecolor=BG_DARK)
fig.suptitle('EFP Term Structure — Annualised Bid / Ask (%)',
             color=C_TEXT, fontsize=13, fontweight='bold')
axes = axes.flatten()

for idx, metal in enumerate(METALS):
    ax = axes[idx]
    df = efp_data[metal]
    color = METAL_COLORS[metal]
    name  = METAL_NAMES[metal]

    contracts = df.index.tolist()
    x = list(range(len(contracts)))

    ann_bid = df['Ann Bid (%)'].tolist()
    ann_ask = df['Ann Ask (%)'].tolist()
    ann_mid = df['Ann Mid (%)'].tolist()
    days_v  = df['Days'].tolist()

    # Fill bid/ask band
    valid = [(i, b, a, m) for i, (b, a, m) in enumerate(zip(ann_bid, ann_ask, ann_mid))
             if not (isinstance(b, float) and np.isnan(b))]

    if valid:
        xi   = [v[0] for v in valid]
        bids = [v[1] for v in valid]
        asks = [v[2] for v in valid]
        mids = [v[3] for v in valid]

        ax.fill_between(xi, bids, asks, color=color, alpha=0.18, label='Bid/Ask band')
        ax.plot(xi, bids, color=color, alpha=0.6, lw=1.2, ls='--', marker='v', ms=5, label='Ann EFP Bid')
        ax.plot(xi, asks, color=color, alpha=0.6, lw=1.2, ls='--', marker='^', ms=5, label='Ann EFP Ask')
        ax.plot(xi, mids, color=color, lw=2,   marker='o', ms=6, label='Ann EFP Mid')

        # Annotate mid values
        for i, (m, d) in enumerate(zip(mids, [days_v[j] for j in xi])):
            d_str = f'\n({d}d)' if isinstance(d, int) else ''
            ax.text(i, m, f'{m:+.3f}%{d_str}',
                    ha='center', va='bottom' if m >= 0 else 'top',
                    color=C_TEXT, fontsize=7.5)

    ax.axhline(0, color=C_MUTED, lw=0.8, ls=':')
    ax.set_title(f'{name} ({metal})', color=color, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(contracts)
    ax.set_ylabel('Ann. EFP (%)', color=C_MUTED)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:+.3f}%'))
    ax.legend(fontsize=7, loc='best')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELL 6 — EFP Bid/Ask Spread Across Metals ($/oz)
#  Wider spread = less liquid / higher transaction cost
# ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor=BG_DARK)
fig.suptitle('EFP Bid/Ask Spread Comparison', color=C_TEXT, fontsize=12, fontweight='bold')

# ── Panel 1: spread in $/oz per contract ─────────────────────
ax1 = axes[0]
x    = np.arange(len(CONTRACTS))
w    = 0.18
offsets = np.linspace(-(len(METALS)-1)*w/2, (len(METALS)-1)*w/2, len(METALS))

for i, metal in enumerate(METALS):
    spreads = efp_data[metal]['EFP Spread'].tolist()
    ax1.bar(x + offsets[i], spreads, width=w,
            color=METAL_COLORS[metal], alpha=0.8, label=METAL_NAMES[metal])

ax1.set_xticks(x); ax1.set_xticklabels(CONTRACTS)
ax1.set_title('EFP Spread (Ask − Bid, $/oz)', color=C_TEXT)
ax1.set_ylabel('Spread ($/oz)', color=C_MUTED)
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3, axis='y')

# ── Panel 2: mid EFP across metals at C1 ─────────────────────
ax2 = axes[1]
for metal in METALS:
    df   = efp_data[metal]
    mids = df['Ann Mid (%)'].tolist()
    xi   = list(range(len(CONTRACTS)))
    ax2.plot(xi, mids, color=METAL_COLORS[metal], marker='o',
             lw=2, ms=6, label=f'{METAL_NAMES[metal]} mid')

ax2.axhline(0, color=C_MUTED, lw=0.8, ls=':')
ax2.set_xticks(xi); ax2.set_xticklabels(CONTRACTS)
ax2.set_title('Ann. EFP Mid (%) — All Metals', color=C_TEXT)
ax2.set_ylabel('Ann. EFP (%)', color=C_MUTED)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:+.3f}%'))
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Timestamp ─────────────────────────────────────────────────
print(f"\nSnapshot: {datetime.now().strftime('%Y-%m-%d  %H:%M:%S')}")
print("To refresh: re-run cells 3 → 6")
